# 02 transform

Transformação e limpeza dos dados

In [17]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from config import engine
from sqlalchemy import text

print("Conexão pronta para uso.")

Conexão pronta para uso.


In [18]:
import pandas as pd

# Carregar os dados dos pedidos
EXCEL_PATH = '../data/raw/Sample - Superstore.xls'
orders = pd.read_excel(EXCEL_PATH, sheet_name='Orders', engine='xlrd')

# Extrair todas as datas únicas dos pedidos
all_dates = pd.to_datetime(orders['Order Date'])
dates = pd.date_range(all_dates.min(), all_dates.max(), freq='D')
dim_tempo = pd.DataFrame({
'sk_tempo' : range(1, len(dates)+1),
'data_completa': dates.strftime('%Y-%m-%d'),
'dia' : dates.day,
'mes' : dates.month,
'nome_mes' : dates.strftime('%B'),
'trimestre' : dates.quarter,
'ano' : dates.year,
'semana_ano' : dates.isocalendar().week.astype(int),
'dia_semana' : dates.dayofweek + 1,
'nome_dia' : dates.strftime('%A'),
'flag_fds' : (dates.dayofweek >= 5).astype(int),
})
print(f'DIM_TEMPO: {len(dim_tempo)} datas | {dim_tempo.ano.nunique()} anos ({dim_tempo.ano.min()}-{dim_tempo.ano.max()})')

DIM_TEMPO: 1458 datas | 4 anos (2014-2017)


In [19]:
# Extrair produtos únicos do Excel
dim_produto_raw = orders[['Product ID', 'Product Name', 'Category', 'Sub-Category']].drop_duplicates('Product ID').copy()

# Verificar inconsistências: mesmo Product ID com nomes diferentes?
inconsist = orders.groupby('Product ID')['Category'].nunique()
print(f'Produtos com categoria inconsistente: {(inconsist > 1).sum()}')

# Renomear e gerar surrogate key
dim_produto = dim_produto_raw.rename(columns={
    'Product ID' : 'nk_produto',
    'Product Name': 'nome_produto',
    'Category' : 'categoria',
    'Sub-Category': 'subcategoria',
})

dim_produto.insert(0, 'sk_produto', range(1, len(dim_produto)+1))
print(f'DIM_PRODUTO: {len(dim_produto)} produtos | {dim_produto.categoria.nunique()} categorias | {dim_produto.subcategoria.nunique()} subcategorias')

Produtos com categoria inconsistente: 0
DIM_PRODUTO: 1862 produtos | 3 categorias | 17 subcategorias


In [20]:
# Clientes únicos (mesma pessoa pode aparecer em múltiplos pedidos)
dim_cliente_raw = orders[['Customer ID','Customer Name','Segment',
'City','State','Region','Country']
].drop_duplicates('Customer ID').copy()
# Limpeza: padronizar State (verificar variações)
print('States únicos:', dim_cliente_raw['State'].nunique())
print(dim_cliente_raw['State'].sort_values().unique()[:10])
dim_cliente = dim_cliente_raw.rename(columns={
'Customer ID' : 'nk_cliente',
'Customer Name': 'nome_cliente',
'Segment' : 'segmento',
'City' : 'cidade',
'State' : 'estado',
'Region' : 'regiao',
'Country' : 'pais',
})
dim_cliente.insert(0, 'sk_cliente', range(1, len(dim_cliente)+1))
print(f'DIM_CLIENTE: {len(dim_cliente)} clientes |{dim_cliente.segmento.nunique()} segmentos')

States únicos: 41
<StringArray>
[             'Alabama',              'Arizona',             'Arkansas',
           'California',             'Colorado',          'Connecticut',
             'Delaware', 'District of Columbia',              'Florida',
              'Georgia']
Length: 10, dtype: str
DIM_CLIENTE: 793 clientes |3 segmentos


In [21]:
import numpy as np

# Calcular prazo real de entrega
orders['order_dt'] = pd.to_datetime(orders['Order Date'])
orders['ship_dt'] = pd.to_datetime(orders['Ship Date'])
orders['prazo_dias'] = (orders['ship_dt'] - orders['order_dt']).dt.days

# Dimensão de envio — combinações únicas de Ship Mode + prazo
dim_envio_raw = orders[['Ship Mode']].drop_duplicates().copy()

# Enriquecer com prazo médio por Ship Mode
prazo_medio = orders.groupby('Ship Mode')['prazo_dias'].mean().round(1).reset_index()
prazo_medio.columns = ['Ship Mode','prazo_medio_dias']

dim_envio = dim_envio_raw.merge(prazo_medio, on='Ship Mode')
dim_envio = dim_envio.rename(columns={'Ship Mode':'modalidade_envio'})
dim_envio.insert(0, 'sk_envio', range(1, len(dim_envio)+1))

print(dim_envio.to_string(index=False))

 sk_envio modalidade_envio  prazo_medio_dias
        1     Second Class               3.2
        2   Standard Class               5.0
        3      First Class               2.2
        4         Same Day               0.0


In [22]:
# Mapas de surrogate keys
sk_cli = dim_cliente.set_index('nk_cliente')['sk_cliente']
sk_pro = dim_produto.set_index('nk_produto')['sk_produto']
sk_tmp = dim_tempo.set_index('data_completa')['sk_tempo']
sk_env = dict(zip(dim_envio['modalidade_envio'], dim_envio['sk_envio']))

orders['data_str'] = orders['order_dt'].dt.strftime('%Y-%m-%d')

fato = orders.copy()
fato['sk_cliente'] = fato['Customer ID'].map(sk_cli)
fato['sk_produto'] = fato['Product ID'].map(sk_pro)
fato['sk_tempo'] = fato['data_str'].map(sk_tmp)
fato['sk_envio'] = fato['Ship Mode'].map(sk_env)

returns = pd.read_excel(EXCEL_PATH, sheet_name='Returns', engine='xlrd')

returned_ids = set(returns['Order ID'].astype(str))
fato['flag_retorno'] = fato['Order ID'].isin(returned_ids).astype(int)
fato_final = fato[['Order ID','sk_cliente','sk_produto','sk_tempo','sk_envio', 'Sales','Profit','Discount','Quantity','flag_retorno']].copy()
fato_final.columns = ['nk_pedido','sk_cliente','sk_produto','sk_tempo','sk_envio', 'vlr_vendas','vlr_lucro','vlr_desconto','qtd','flag_retorno']

fato_final = fato_final.dropna(subset=['sk_cliente','sk_produto','sk_tempo','sk_envio'])
for col in ['sk_cliente','sk_produto','sk_tempo','sk_envio','qtd','flag_retorno']:
    fato_final[col] = fato_final[col].astype(int)
print(f'FATO_VENDAS: {len(fato_final):,} registros | Retornos: {fato_final.flag_retorno.sum()}')

FATO_VENDAS: 9,994 registros | Retornos: 800


In [23]:
# 1. Cria a DIM_GERENTE a partir da aba 'People'
people = pd.read_excel(EXCEL_PATH, sheet_name='People', engine='xlrd')
dim_gerente = people.rename(columns={'Person': 'nome_gerente', 'Region': 'regiao'})
dim_gerente.insert(0, 'sk_gerente', range(1, len(dim_gerente) + 1))

# 2. No mapeamento da FATO_VENDAS, adiciona a   sk_gerente
sk_ger = dict(zip(dim_gerente['regiao'], dim_gerente['sk_gerente']))
fato['sk_gerente'] = fato['Region'].map(sk_ger)

# Atualiza a lista de colunas finais da fato (adicionando sk_gerente)
fato_final = fato[['Order ID','sk_cliente','sk_produto','sk_tempo','sk_envio', 'sk_gerente', 'Sales','Profit','Discount','Quantity','flag_retorno']].copy()
fato_final.columns = ['nk_pedido','sk_cliente','sk_produto','sk_tempo','sk_envio', 'sk_gerente', 'vlr_vendas','vlr_lucro','vlr_desconto','qtd','flag_retorno']
